In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv("train.csv")

df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

X = df[[
    "Pclass", "Sex", "Age", "SibSp", "Parch",
    "Fare", "Embarked", "FamilySize", "IsAlone"
]]
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numerical_features = [
    "Pclass", "Age", "SibSp", "Parch", "Fare",
    "FamilySize", "IsAlone"
]

categorical_features = ["Sex", "Embarked"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Pipeline Accuracy:", accuracy)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

X_baseline = df[[
    "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"
]]

X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_baseline, y, test_size=0.2, random_state=42, stratify=y
)

baseline_numerical = [
    "Pclass", "Age", "SibSp", "Parch", "Fare"
]

baseline_categorical = ["Sex", "Embarked"]

baseline_preprocessor = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        baseline_numerical
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first"))
        ]),
        baseline_categorical
    )
])

baseline_pipeline = Pipeline([
    ("preprocessing", baseline_preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

baseline_pipeline.fit(X_train_base, y_train_base)

baseline_pred = baseline_pipeline.predict(X_test_base)

baseline_accuracy = accuracy_score(y_test_base, baseline_pred)

print("\nBaseline Accuracy:", baseline_accuracy)
print("Pipeline + Features Accuracy:", accuracy)
print("Improvement:", accuracy - baseline_accuracy)

#using joblib
joblib.dump(pipeline, "titanic_final_pipeline.pkl")

loaded_pipeline = joblib.load("titanic_final_pipeline.pkl")
loaded_predictions = loaded_pipeline.predict(X_test)

print(
    "\nLoaded Pipeline Accuracy:",
    accuracy_score(y_test, loaded_predictions)
)

Pipeline Accuracy: 0.8044692737430168

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.88      0.85       110
           1       0.78      0.68      0.73        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179


Baseline Accuracy: 0.8044692737430168
Pipeline + Features Accuracy: 0.8044692737430168
Improvement: 0.0

Loaded Pipeline Accuracy: 0.8044692737430168
